<a href="https://colab.research.google.com/github/AleksandraNovo/GP5/blob/main/notebooks/02_CNN_condition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. CNN - Классификация состояния одежды по изображению

**Цель:** обучить свёрточную нейронную сеть (CNN) для классификации состояния (condition) секонд-хенд одежды на основе изображений.

**Данные:** [`wargoninnovation/clothingdatasetsecondhand`](https://huggingface.co/datasets/wargoninnovation/clothingdatasetsecondhand) (Hugging Face Datasets)  
> Данные загружаются напрямую через `datasets` библиотеку - локально не хранятся из-за ограничений GitHub.


**Модели:** хранятся в models/resnet50_best.pt

In [ ]:
from datasets import load_dataset

ds = load_dataset("wargoninnovation/clothingdatasetsecondhand")

In [ ]:
ds["train"].column_names

In [ ]:
ds["train"][0]

In [ ]:
ds = ds["train"]

In [ ]:
ds = ds.select_columns(["image", "condition", "type"])

In [ ]:
sample = ds[0]

print(f"condition: {sample['condition']}, type: {sample['type']}")
sample['image']

In [ ]:
from collections import Counter

labels = [sample['condition'] for sample in ds]
counts = Counter(labels)

counts

Видно сильный дисбаланс - класс 1 встречается в 8 раз реже класса 3. Мы потеряем много данных если сократим все до 1233 (30к -> 6к). Cохраним больше данных (4-5к), класс 1 немного дополним аугментациями позже.

In [ ]:
import pandas as pd

target_count = 4500

df = ds.to_pandas()[["condition"]].copy()
df["idx"] = df.index

balanced_idx = (df.groupby("condition", group_keys=False).apply(lambda x: x.sample(min(len(x), target_count), random_state=42))["idx"].tolist())

balanced_ds = ds.select(balanced_idx)

In [ ]:
Counter(balanced_ds["condition"])

In [ ]:
Counter(balanced_ds["type"])

Ситуация, если мы берем еще и целевую переменную type, действительно сложная - 34 класса, из которых многие экстремально маленькие (class 28: 1 объект, class 33: 2 объекта). Это делает задачу классификации type практически нереальной в текущем виде.

In [ ]:
Counter(ds["type"])

Разобьем датасет на 3 части в соотношении 70 / 20 / 10 - train / val / test

In [ ]:
split_1 = balanced_ds.train_test_split(test_size=0.3, seed=42)
train_ds = split_1["train"]

split_2 = split_1["test"].train_test_split(test_size=0.333, seed=42)
val_ds = split_2["train"]
test_ds = split_2["test"]

print(len(train_ds), len(val_ds), len(test_ds))

Расширим обучающую выборку аугментацией

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Для val и test — только нормализация, никаких случайных преобразований

In [ ]:
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ClothingDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image'].convert('RGB')
        label = sample['condition'] - 1  # мы переводим condition из диапазона 1–5 в 0–4, потому что PyTorch ожидает метки начиная с нуля

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = ClothingDataset(train_ds, transform=train_transforms)
val_dataset = ClothingDataset(val_ds, transform=val_test_transforms)
test_dataset = ClothingDataset(test_ds, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape, labels)

1. Архитектура которую будем пробовать первую - **ResNet-50**

In [ ]:
import torch.nn as nn
from torchvision import models

def build_resnet50(num_classes=5):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model

In [ ]:
model = build_resnet50()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

device

У нас задача - порядковой классификации, так как наши классы condition 1–5 - упорядоченные. Обычный CrossEntropyLoss этого не учитывает. Реализуем CORN loss, которая разбивает задачу на 4 бинарных вопроса - "condition > i?" для i = 0, 1, 2, 3 - и учитывает порядок явно.

In [ ]:
def corn_loss(logits, labels, num_classes=5):
    loss = torch.zeros(1, device=logits.device)
    num_examples = 0

    for i in range(num_classes - 1):
        mask = labels >= i
        if mask.sum() == 0:
            continue
        logit_i = logits[mask, i]
        binary_label = (labels[mask] > i).float()
        loss += nn.functional.binary_cross_entropy_with_logits(
            logit_i, binary_label, reduction='sum')
        num_examples += mask.sum()

    return loss / num_examples

In [ ]:
def predict_condition(logits):
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1)

In [ ]:
def build_resnet50(num_classes=5):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes - 1)
    return model

model = build_resnet50()
model = model.to(device)

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = corn_loss(logits, labels)
        loss.backward()
        optimizer.step()

        preds = predict_condition(logits)
        total_loss += loss.item()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total

Будем выводить еще mae , так как эта метрика более показательна чем accuracy

In [ ]:
def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = corn_loss(logits, labels)

            preds = predict_condition(logits)
            total_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    mae = torch.tensor(all_preds).float().sub(torch.tensor(all_labels).float()).abs().mean().item()
    return total_loss / len(loader), correct / total, mae

In [ ]:
def build_resnet50(num_classes=5):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes - 1)
    )
    return model

In [ ]:
def unfreeze_backbone(model, layers=2):

    blocks = [model.layer3, model.layer4][-layers:]

    for block in blocks:
        for param in block.parameters():
            param.requires_grad = True

In [ ]:
unfreeze_backbone(model, layers=2)

optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [ ]:
model = build_resnet50().to(device)

In [ ]:
#model.load_state_dict(torch.load("models/resnet50_best.pt"))

unfreeze_backbone(model, layers=2)

optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

# Логгирование с помощью Mlflow

In [ ]:
!pip install mlflow pyngrok -q

In [ ]:
import mlflow
import mlflow.pytorch

mlflow.set_experiment("GP5_CNN_condition")

In [ ]:
import os
os.makedirs("models", exist_ok=True)

In [ ]:
import numpy as np

EPOCHS = 20
ES_PATIENCE = 5
LR = 3e-4
BATCH_SIZE = 32
DROPOUT = 0.4
TARGET_COUNT = 4500

best_val_mae = np.inf
no_improve = 0

In [ ]:
#model.load_state_dict(torch.load("models/resnet50_best.pt"))

In [ ]:
with mlflow.start_run(run_name="resnet50_corn_loss"):

    mlflow.log_params({
        "model": "resnet50",
        "loss": "CORN",
        "optimizer": "Adam",
        "lr": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "dropout": DROPOUT,
        "target_count": TARGET_COUNT,
        "es_patience": ES_PATIENCE,
    })

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer)
        val_loss, val_acc, val_mae = eval_epoch(model, val_loader)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_mae": val_mae,
            "lr": current_lr,
        }, step=epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | "
              f"val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | "
              f"val_mae: {val_mae:.4f} | lr: {current_lr:.2e}")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            no_improve = 0
            torch.save(model.state_dict(), "models/resnet50_best.pt")
            mlflow.pytorch.log_model(model, artifact_path="best_model")
            print("модель сохранена")
        else:
            no_improve += 1
            if no_improve >= ES_PATIENCE:
                print("early stopping!")
                break

    mlflow.log_metrics({
        "best_val_mae": best_val_mae,
        "total_epochs": epoch + 1,
    })

In [ ]:
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = predict_condition(model(images))
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
mae = mean_absolute_error(all_labels, all_preds)

print(f"Accuracy: {acc:.4f}, MAE: {mae:.4f}")
print(classification_report(all_labels, all_preds, target_names=[f'cond {i}' for i in range(1, 6)]))

Далее предскажем для всех картинок наш целевой признак и будем в дальнейшем использовать его для обучении регрессии.

In [ ]:
full_dataset = ClothingDataset(ds, transform=val_test_transforms)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for images, _ in full_loader:
        images = images.to(device)
        preds = predict_condition(model(images))
        all_preds.extend(preds.cpu().numpy())

In [ ]:
df = ds.to_pandas().drop(columns=["image"])
df["condition_predicted"] = [p + 1 for p in all_preds]

df.to_csv("clothing_table.csv", index=False)

In [ ]:
print(df.shape)
df.head()

По плану и критериям еще 2 архитектуры:
- EfficientNet-B3 (Другой принцип масштабирования, меньше параметров, часто точнее)
- ViT-B/16 (Принципиально другая архитектура - не CNN, а self-attention; не разбирался на лекциях - +баллы по критерию 4)

# EfficientNet-B3

In [ ]:
from torchvision import models

def build_efficientnet_b3(num_classes=5):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes - 1)
    )
    return model

In [ ]:
model_eff = build_efficientnet_b3().to(device)

optimizer_eff = Adam(filter(lambda p: p.requires_grad, model_eff.parameters()), lr=3e-4)
scheduler_eff = ReduceLROnPlateau(optimizer_eff, mode='min', patience=3, factor=0.5)

In [ ]:
def unfreeze_efficientnet(model, last_n_blocks=2):
    blocks = list(model.features.children())
    for block in blocks[-last_n_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

In [ ]:
import numpy as np

EPOCHS = 20
ES_PATIENCE = 5
best_val_mae_eff = np.inf
no_improve_eff = 0

os.makedirs("models", exist_ok=True)

with mlflow.start_run(run_name="efficientnet_b3_corn_loss"):

    mlflow.log_params({
        "model": "efficientnet_b3",
        "loss": "CORN",
        "optimizer": "Adam",
        "lr": 3e-4,
        "epochs": EPOCHS,
        "batch_size": 32,
        "dropout": 0.4,
        "phase": "head_only"
    })

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model_eff, train_loader, optimizer_eff)
        val_loss, val_acc, val_mae = eval_epoch(model_eff, val_loader)
        scheduler_eff.step(val_loss)
        current_lr = optimizer_eff.param_groups[0]['lr']

        mlflow.log_metrics({
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "val_mae": val_mae, "lr": current_lr,
        }, step=epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss: {train_loss:.4f} | "
              f"train_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | "
              f"val_acc: {val_acc:.4f} | val_mae: {val_mae:.4f} | lr: {current_lr:.2e}")

        if val_mae < best_val_mae_eff:
            best_val_mae_eff = val_mae
            no_improve_eff = 0
            torch.save(model_eff.state_dict(), "models/efficientnet_b3_best.pt")
            mlflow.pytorch.log_model(model_eff, artifact_path="best_model")
            print("модель сохранена")
        else:
            no_improve_eff += 1
            if no_improve_eff >= ES_PATIENCE:
                print("early stopping!")
                break

    mlflow.log_metrics({"best_val_mae": best_val_mae_eff})

# Fine-tuning (разморозка последних блоков)
Загружаем лучшие веса, размораживаем последние 2 блока и дообучаем с очень маленьким lr, чтобы не сломать ImageNet-признаки.

In [ ]:
model_eff.load_state_dict(torch.load("models/efficientnet_b3_best.pt"))

unfreeze_efficientnet(model_eff, last_n_blocks=2)

optimizer_eff = Adam(filter(lambda p: p.requires_grad, model_eff.parameters()), lr=1e-5)
scheduler_eff = ReduceLROnPlateau(optimizer_eff, mode='min', patience=3, factor=0.5)

In [ ]:
best_val_mae_eff = np.inf
no_improve_eff = 0

with mlflow.start_run(run_name="efficientnet_b3_finetune"):
    for epoch in range(10):
        train_loss, train_acc = train_epoch(model_eff, train_loader, optimizer_eff)
        val_loss, val_acc, val_mae = eval_epoch(model_eff, val_loader)
        scheduler_eff.step(val_loss)

        print(f"FT Epoch {epoch+1}/10 | val_mae: {val_mae:.4f}")

        if val_mae < best_val_mae_eff:
            best_val_mae_eff = val_mae
            torch.save(model_eff.state_dict(), "models/efficientnet_b3_best.pt")
            print("модель сохранена")
        else:
            no_improve_eff += 1
            if no_improve_eff >= 3:
                print("early stopping!")
                break

In [ ]:
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report, confusion_matrix

model_eff.load_state_dict(torch.load("models/efficientnet_b3_best.pt"))
model_eff.eval()

all_preds_eff, all_labels_eff = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = predict_condition(model_eff(images))
        all_preds_eff.extend(preds.cpu().numpy())
        all_labels_eff.extend(labels.cpu().numpy())

all_preds_eff = np.array(all_preds_eff)
all_labels_eff = np.array(all_labels_eff)

acc_eff = accuracy_score(all_labels_eff, all_preds_eff)
mae_eff = mean_absolute_error(all_labels_eff, all_preds_eff)

print(f"EfficientNet-B3 | Accuracy: {acc_eff:.4f}, MAE: {mae_eff:.4f}")
print(classification_report(all_labels_eff, all_preds_eff,
      target_names=[f'cond {i}' for i in range(1, 6)]))

In [ ]:
full_dataset = ClothingDataset(ds, transform=val_test_transforms)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
model_eff.load_state_dict(torch.load("models/efficientnet_b3_best.pt"))
model_eff.eval()

all_preds_eff_full = []

with torch.no_grad():
    for images, _ in full_loader:
        images = images.to(device)
        preds = predict_condition(model_eff(images))
        all_preds_eff_full.extend(preds.cpu().numpy())

In [ ]:
df_eff = ds.to_pandas().drop(columns=["image"])
df_eff["condition"] = [p + 1 for p in all_preds_eff_full]

df_eff.to_csv("clothing_table.csv", index=False)

print(df_eff.shape)
df_eff.head()

# Модель ViT-B/16

In [ ]:
def build_vit_b16(num_classes=5):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes - 1)
    )
    return model

In [ ]:
model_vit = build_vit_b16().to(device)

optimizer_vit = Adam(filter(lambda p: p.requires_grad, model_vit.parameters()), lr=3e-4)
scheduler_vit = ReduceLROnPlateau(optimizer_vit, mode='min', patience=3, factor=0.5)

In [ ]:
def unfreeze_vit(model, last_n_blocks=2):
    encoder_blocks = list(model.encoder.layers.children())
    for block in encoder_blocks[-last_n_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

In [55]:
best_val_mae_vit = np.inf
no_improve_vit = 0

with mlflow.start_run(run_name="vit_b16_corn_loss"):

    mlflow.log_params({
        "model": "vit_b16",
        "loss": "CORN",
        "optimizer": "Adam",
        "lr": 3e-4,
        "epochs": EPOCHS,
        "batch_size": 32,
        "dropout": 0.4,
        "phase": "head_only"
    })

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model_vit, train_loader, optimizer_vit)
        val_loss, val_acc, val_mae = eval_epoch(model_vit, val_loader)
        scheduler_vit.step(val_loss)
        current_lr = optimizer_vit.param_groups[0]['lr']

        mlflow.log_metrics({
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "val_mae": val_mae, "lr": current_lr,
        }, step=epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss: {train_loss:.4f} | "
              f"train_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | "
              f"val_acc: {val_acc:.4f} | val_mae: {val_mae:.4f} | lr: {current_lr:.2e}")

        if val_mae < best_val_mae_vit:
            best_val_mae_vit = val_mae
            no_improve_vit = 0
            torch.save(model_vit.state_dict(), "models/vit_b16_best.pt")
            mlflow.pytorch.log_model(model_vit, artifact_path="best_model")
            print("модель сохранена")
        else:
            no_improve_vit += 1
            if no_improve_vit >= ES_PATIENCE:
                print("early stopping!")
                break

    mlflow.log_metrics({"best_val_mae": best_val_mae_vit})

Epoch 1/20 | train_loss: 0.5138 | train_acc: 0.2442 | val_loss: 0.4940 | val_acc: 0.2588 | val_mae: 1.3480 | lr: 3.00e-04


2026/06/16 08:05:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:05:13 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 08:05:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 08:05:22 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

модель сохранена
Epoch 2/20 | train_loss: 0.4972 | train_acc: 0.2597 | val_loss: 0.4904 | val_acc: 0.2716 | val_mae: 1.2827 | lr: 3.00e-04


2026/06/16 08:09:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:09:12 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 08:09:19 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 08:09:26 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

модель сохранена
Epoch 3/20 | train_loss: 0.4918 | train_acc: 0.2630 | val_loss: 0.4905 | val_acc: 0.2633 | val_mae: 1.4389 | lr: 3.00e-04
Epoch 4/20 | train_loss: 0.4915 | train_acc: 0.2685 | val_loss: 0.4876 | val_acc: 0.2783 | val_mae: 1.2994 | lr: 3.00e-04
Epoch 5/20 | train_loss: 0.4889 | train_acc: 0.2707 | val_loss: 0.4880 | val_acc: 0.2664 | val_mae: 1.3204 | lr: 3.00e-04
Epoch 6/20 | train_loss: 0.4868 | train_acc: 0.2717 | val_loss: 0.4874 | val_acc: 0.2682 | val_mae: 1.3794 | lr: 3.00e-04
Epoch 7/20 | train_loss: 0.4877 | train_acc: 0.2686 | val_loss: 0.4870 | val_acc: 0.2755 | val_mae: 1.2448 | lr: 3.00e-04


2026/06/16 08:28:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:28:16 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 08:28:24 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 08:28:31 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

модель сохранена
Epoch 8/20 | train_loss: 0.4865 | train_acc: 0.2764 | val_loss: 0.4877 | val_acc: 0.2814 | val_mae: 1.3412 | lr: 3.00e-04
Epoch 9/20 | train_loss: 0.4847 | train_acc: 0.2756 | val_loss: 0.4866 | val_acc: 0.2781 | val_mae: 1.3589 | lr: 3.00e-04
Epoch 10/20 | train_loss: 0.4852 | train_acc: 0.2744 | val_loss: 0.4886 | val_acc: 0.2781 | val_mae: 1.4002 | lr: 3.00e-04
Epoch 11/20 | train_loss: 0.4852 | train_acc: 0.2732 | val_loss: 0.4861 | val_acc: 0.2799 | val_mae: 1.3033 | lr: 3.00e-04
Epoch 12/20 | train_loss: 0.4829 | train_acc: 0.2781 | val_loss: 0.4867 | val_acc: 0.2734 | val_mae: 1.4433 | lr: 3.00e-04
early stopping!


# Fine-tuning
ViT чувствителен к большому lr при разморозке - ставим 5e-6, чуть меньше чем для EfficientNet.

In [56]:
model_vit.load_state_dict(torch.load("models/vit_b16_best.pt"))

unfreeze_vit(model_vit, last_n_blocks=2)

optimizer_vit = Adam(
    filter(lambda p: p.requires_grad, model_vit.parameters()),
    lr=5e-6
)
scheduler_vit = ReduceLROnPlateau(optimizer_vit, mode='min', patience=3, factor=0.5)

best_val_mae_vit = np.inf
no_improve_vit = 0

with mlflow.start_run(run_name="vit_b16_finetune"):

    mlflow.log_params({
        "model": "vit_b16",
        "phase": "finetune",
        "lr": 5e-6,
        "unfrozen_blocks": 2,
    })

    for epoch in range(10):
        train_loss, train_acc = train_epoch(model_vit, train_loader, optimizer_vit)
        val_loss, val_acc, val_mae = eval_epoch(model_vit, val_loader)
        scheduler_vit.step(val_loss)
        current_lr = optimizer_vit.param_groups[0]['lr']

        mlflow.log_metrics({
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "val_mae": val_mae, "lr": current_lr,
        }, step=epoch)

        print(f"FT Epoch {epoch+1}/10 | val_mae: {val_mae:.4f} | lr: {current_lr:.2e}")

        if val_mae < best_val_mae_vit:
            best_val_mae_vit = val_mae
            no_improve_vit = 0
            torch.save(model_vit.state_dict(), "models/vit_b16_best.pt")
            mlflow.pytorch.log_model(model_vit, artifact_path="best_model")
            print("модель сохранена")
        else:
            no_improve_vit += 1
            if no_improve_vit >= 3:
                print("early stopping!")
                break

    mlflow.log_metrics({"best_val_mae": best_val_mae_vit})

FT Epoch 1/10 | val_mae: 1.2659 | lr: 5.00e-06


2026/06/16 08:52:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:52:08 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 08:52:16 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 08:52:23 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

модель сохранена
FT Epoch 2/10 | val_mae: 1.3867 | lr: 5.00e-06
FT Epoch 3/10 | val_mae: 1.2814 | lr: 5.00e-06
FT Epoch 4/10 | val_mae: 1.3646 | lr: 5.00e-06
early stopping!


In [57]:
model_vit.load_state_dict(torch.load("models/vit_b16_best.pt"))
model_vit.eval()

all_preds_vit, all_labels_vit = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = predict_condition(model_vit(images))
        all_preds_vit.extend(preds.cpu().numpy())
        all_labels_vit.extend(labels.cpu().numpy())

all_preds_vit = np.array(all_preds_vit)
all_labels_vit = np.array(all_labels_vit)

acc_vit = accuracy_score(all_labels_vit, all_preds_vit)
mae_vit = mean_absolute_error(all_labels_vit, all_preds_vit)

print(f"ViT-B/16 | Accuracy: {acc_vit:.4f}, MAE: {mae_vit:.4f}")
print(classification_report(all_labels_vit, all_preds_vit,
      target_names=[f'cond {i}' for i in range(1, 6)]))

ViT-B/16 | Accuracy: 0.2648, MAE: 1.2877
              precision    recall  f1-score   support

      cond 1       0.00      0.00      0.00       129
      cond 2       0.00      0.00      0.00       477
      cond 3       0.31      0.03      0.06       433
      cond 4       0.25      0.62      0.36       470
      cond 5       0.28      0.49      0.36       413

    accuracy                           0.26      1922
   macro avg       0.17      0.23      0.16      1922
weighted avg       0.19      0.26      0.18      1922



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [58]:
model_vit.load_state_dict(torch.load("models/vit_b16_best.pt"))
model_vit.eval()

all_preds_vit_full = []

with torch.no_grad():
    for images, _ in full_loader:
        images = images.to(device)
        preds = predict_condition(model_vit(images))
        all_preds_vit_full.extend(preds.cpu().numpy())

df_vit = ds.to_pandas().drop(columns=["image"])
df_vit["condition_predicted"] = [p + 1 for p in all_preds_vit_full]

df_vit.to_csv("clothing_table_vit.csv", index=False)

print(df_vit.shape)
df_vit.head()

(30192, 3)


,condition,type,condition_predicted
0,2,14,4
1,3,20,5
2,3,14,4
3,2,7,4
4,5,20,4
